# 18 — Diagnóstico de treino e margens correspondentes

Reajusta ExtraTrees nas duas janelas e o ensemble de cinco componentes, sem treinar com alvos de 2026. Usa consumos passados de 2026 somente antes de cada origem. Recalcula margens com resíduos históricos do mesmo modelo e configuração. Os resultados são diagnósticos após o exame da avaliação de 2026. As emissões cegas iniciais e as janelas congeladas de setembro e outubro são preservadas.

In [1]:
from pathlib import Path
from datetime import datetime
import hashlib
import importlib
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path(r"F:\PRODUTO_MESTRADO")
SCRIPTS = ROOT / "scripts"
RESULTS = ROOT / "resultados" / "diagnostico_2026_auditado_20261003"
FIG_DIR = ROOT / "resultados" / "figuras_dissertacao"
RESULTS.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
AS_OF = pd.Timestamp("2026-10-02")
ACTUAL_PATH = ROOT / "data" / "validacao_2026_restrita" / "consumo_real_2026.csv"
SCENARIO_PATH = ROOT / "data" / "gold" / "cenarios_previsao_consumo_2026.csv"
BASE_FEATURES_PATH = ROOT / "data" / "experimentos" / "consumo_cardapio_calendario" / "features" / "features_supervisionadas_horizonte_7d.csv"
RES_FEATURES_PATH = ROOT / "data" / "experimentos" / "reservas_consumo_cardapio_calendario_clima" / "features" / "features_supervisionadas_horizonte_7d.csv"

sys.path.insert(0, str(SCRIPTS))
import feature_engineering_temporal as fe_base
import competir_modelos_temporalmente as comp_base

gold = fe_base.read_gold()
gold["data"] = pd.to_datetime(gold["data"], errors="raise").dt.normalize()
calendar = fe_base.read_calendar()
calendar["data"] = pd.to_datetime(calendar["data"], errors="raise").dt.normalize()

raw_actual = pd.read_csv(ACTUAL_PATH, sep=None, engine="python", encoding="utf-8-sig", dtype=str, keep_default_na=False)
raw_actual["data"] = pd.to_datetime(raw_actual["data"], dayfirst=True, errors="coerce", format="mixed").dt.normalize()
raw_actual["consumo"] = pd.to_numeric(raw_actual["consumo"].replace("", np.nan), errors="coerce")
actual = raw_actual.dropna(subset=["data", "consumo"]).loc[lambda x: x["data"].dt.year.eq(2026)].copy()
if actual["data"].duplicated().any():
    raise ValueError("Há mais de um consumo real por data em 2026.")
if actual["data"].max() > AS_OF:
    raise AssertionError("Há consumo de 2026 posterior à origem de previsão.")

# Os valores reais de 2026 só são inseridos em uma cópia em memória para
# simular o histórico já observado antes de cada origem e para pontuar o teste.
actual_by_date = actual.set_index("data")["consumo"]
gold_rolling = gold.copy()
gold_rolling["consumo_total"] = pd.to_numeric(gold_rolling["consumo_total"], errors="coerce")
if not actual_by_date.index.isin(gold_rolling["data"]).all():
    missing = actual_by_date.index.difference(gold_rolling["data"])
    raise ValueError(f"Datas reais ausentes da Gold: {missing[:5].tolist()}")
gold_rolling.loc[gold_rolling["data"].isin(actual_by_date.index), "consumo_total"] = gold_rolling.loc[
    gold_rolling["data"].isin(actual_by_date.index), "data"
].map(actual_by_date)

observed_max = actual["data"].max()
observed_weekdays = actual.loc[actual["data"].dt.dayofweek.lt(5)].copy()
print(f"Consumos reais usados como histórico/avaliação: {len(actual)} datas; último registro: {observed_max.date()}.")
print("Nenhum alvo 2026 será usado para ajustar os modelos; somente os registros anteriores à origem entram nos atributos de histórico.")

future_inputs = pd.DataFrame({
    "data": pd.to_datetime(["2026-10-05", "2026-10-06", "2026-10-07", "2026-10-08", "2026-10-09"]),
    "quantidade_informada": [225, 220, 220, 210, 200],
})
if future_inputs["data"].min() <= AS_OF:
    raise AssertionError("A semana precisa estar depois da origem de previsão.")

scenario = pd.read_csv(SCENARIO_PATH, encoding="utf-8-sig")
scenario["data"] = pd.to_datetime(scenario["data"], errors="raise").dt.normalize()
scenario_future = scenario.loc[scenario["data"].isin(future_inputs["data"])].copy()
if set(scenario_future["data"]) != set(future_inputs["data"]):
    raise ValueError("Faltam covariáveis de cardápio/calendário para 5–9 de outubro.")

base_features = pd.read_csv(BASE_FEATURES_PATH, encoding="utf-8-sig", low_memory=False)
res_features = pd.read_csv(RES_FEATURES_PATH, encoding="utf-8-sig", low_memory=False)
for table in (base_features, res_features):
    table["data_alvo"] = pd.to_datetime(table["data_alvo"], errors="raise").dt.normalize()
    table["consumo_total"] = pd.to_numeric(table["consumo_total"], errors="coerce")
    if table["data_alvo"].dt.year.max() > 2025:
        raise AssertionError("A tabela supervisionada inclui alvos posteriores a 2025.")

training_sets = {
    "Treino 2023–2025": base_features.loc[base_features["data_alvo"].dt.year.le(2025)].copy(),
    "Treino somente 2025": base_features.loc[base_features["data_alvo"].dt.year.eq(2025)].copy(),
}
res_training_sets = {
    "Treino 2023–2025": res_features.loc[res_features["data_alvo"].dt.year.le(2025)].copy(),
    "Treino somente 2025": res_features.loc[res_features["data_alvo"].dt.year.eq(2025)].copy(),
}
print("Alvos supervisionados por janela:", {k: len(v) for k, v in training_sets.items()})

Consumos reais usados como histórico/avaliação: 123 datas; último registro: 2026-08-31.
Nenhum alvo 2026 será usado para ajustar os modelos; somente os registros anteriores à origem entram nos atributos de histórico.
Alvos supervisionados por janela: {'Treino 2023–2025': 500, 'Treino somente 2025': 173}


In [2]:
# Carrega a engenharia da linha com reservas. O clima é retirado explicitamente dos preditores.
sys.path.insert(0, str(SCRIPTS / "linha_reservas_clima"))
sys.modules.pop("feature_engineering_temporal", None)
sys.modules.pop("competir_modelos_temporalmente", None)
fe_res = importlib.import_module("feature_engineering_temporal")
comp_res = importlib.import_module("competir_modelos_temporalmente")

# Acrescenta em memória os números informados no e-mail da semana de 5–9/10.
read_known_original = fe_res._read_known_exogenous
def read_known_with_weekly_email():
    reservations, climate = read_known_original()
    reservations = reservations.copy()
    for row in future_inputs.itertuples(index=False):
        reservations.loc[pd.Timestamp(row.data), ["reservas_conhecidas", "reserva_disponivel"]] = [
            float(row.quantidade_informada), 1
        ]
    return reservations, climate
fe_res._read_known_exogenous = read_known_with_weekly_email

# Origens de sexta-feira correspondentes aos alvos reais e à próxima semana.
def friday_origin(target):
    target = pd.Timestamp(target).normalize()
    return target - pd.Timedelta(days=target.dayofweek + 3)

scored_targets = pd.DatetimeIndex(observed_weekdays["data"].sort_values().unique())
origins = {friday_origin(d) for d in scored_targets}
origins.add(AS_OF)
folds = pd.DataFrame({"origem": sorted(origins)})
folds["fold"] = np.arange(2001, 2001 + len(folds))
folds = folds[["fold", "origem"]]
print(f"Folds semanais: {len(folds)-1} para pontuação histórica de 2026 e 1 para a semana de 5–9/10.")

climate_prefixes = ("temp_", "precipitacao_", "clima_")
res_predictors = [c for c in fe_res.get_predictor_columns(res_features) if not c.startswith(climate_prefixes)]
assert "reservas_conhecidas" in res_predictors and "reserva_disponivel" in res_predictors

def execute_case(label, features, trainer, predictor_columns=None):
    prediction, failures, audit, weights = trainer.run_competition(
        features,
        gold_rolling,
        calendar,
        folds,
        model_names=["ExtraTrees"],
        predictor_columns=predictor_columns,
        progress_callback=lambda p: print(f"{label}: {p['fold_number']}/{p['fold_count']} origens")
        if p["fold_number"] % 8 == 0 or p["fold_number"] == p["fold_count"] else None,
    )
    result = prediction.loc[prediction["modelo"].eq("ExtraTrees")].copy()
    result["data"] = pd.to_datetime(result["data"], errors="raise").dt.normalize()
    result["origem"] = pd.to_datetime(result["origem"], errors="raise").dt.normalize()
    result["caso"] = label
    if result.empty:
        raise RuntimeError(f"Nenhuma previsão retornada para {label}.")
    if not failures.empty:
        print(f"Falhas registradas em {label}: {len(failures)}")
    return result, failures

case_results = []
failure_results = []
for window_name, train_features in training_sets.items():
    print("Executando sem reservas:", window_name)
    result, failures = execute_case(f"Sem reservas | {window_name}", train_features, comp_base)
    case_results.append(result)
    if not failures.empty:
        failures["caso"] = f"Sem reservas | {window_name}"
        failure_results.append(failures)

for window_name, train_features in res_training_sets.items():
    print("Executando com reservas:", window_name)
    result, failures = execute_case(f"Com reservas | {window_name}", train_features, comp_res, res_predictors)
    case_results.append(result)
    if not failures.empty:
        failures["caso"] = f"Com reservas | {window_name}"
        failure_results.append(failures)

# Reexecuta o mesmo ensemble congelado no notebook 12, agora com consumos
# passados de 2026 disponíveis somente como atributos anteriores à origem.
frozen_manifest = json.loads((ROOT / "resultados" / "validacao_2026" / "manifesto_previsoes_cegas_2026.json").read_text(encoding="utf-8"))
frozen_components = frozen_manifest["componentes"]
ensemble_predictions, ensemble_failures, ensemble_audit, ensemble_weights = comp_base.run_competition(
    training_sets["Treino 2023–2025"],
    gold_rolling,
    calendar,
    folds,
    model_names=frozen_components,
    progress_callback=lambda p: print(f"Ensemble congelado: {p['fold_number']}/{p['fold_count']} origens")
    if p["fold_number"] % 8 == 0 or p["fold_number"] == p["fold_count"] else None,
)
ensemble_only = ensemble_predictions.loc[ensemble_predictions["modelo"].eq("Ensemble_top5_validacao_media")].copy()
if ensemble_only.empty or ensemble_failures["modelo"].isin(frozen_components).any():
    raise RuntimeError("Não foi possível reconstruir todos os componentes do ensemble congelado.")
ensemble_only["data"] = pd.to_datetime(ensemble_only["data"], errors="raise").dt.normalize()
ensemble_only["origem"] = pd.to_datetime(ensemble_only["origem"], errors="raise").dt.normalize()
ensemble_only["caso"] = "Ensemble congelado | Treino 2023–2025"
case_results.append(ensemble_only)

predictions = pd.concat(case_results, ignore_index=True)
if failure_results:
    pd.concat(failure_results, ignore_index=True).to_csv(RESULTS / "falhas_modelos.csv", index=False, encoding="utf-8-sig")
predictions.to_csv(RESULTS / "previsoes_brutas_janela_treino_2026.csv", index=False, encoding="utf-8-sig")
print(f"Previsões de ExtraTrees geradas: {len(predictions):,} linhas.")
source=RESULTS/"previsoes_brutas_janela_treino_2026.csv"
source_hash=hashlib.sha256(source.read_bytes()).hexdigest()
predictions=predictions.loc[predictions['data'].isin(observed_weekdays['data'])].copy()
reserve=pd.read_csv(ROOT/'data/silver/reservas/reservas_diario_2023_2026.csv',parse_dates=['data'])
eligible=set(reserve.loc[reserve.status_reservas.eq('observado') & reserve.reservas.notna(),'data'])
common=set.intersection(*predictions.groupby('caso')['data'].apply(set).tolist()) & eligible
predictions=predictions.loc[predictions['data'].isin(common)].copy()
actual=actual.loc[actual['data'].isin(common)].copy()
assert predictions.groupby('caso')['data'].nunique().eq(len(common)).all()
# Resíduos do mesmo ensemble fixo de cinco componentes com o conjunto completo.
manifest = json.loads((ROOT / "resultados" / "validacao_2026" / "manifesto_previsoes_cegas_2026.json").read_text(encoding="utf-8"))
historical = pd.read_csv(ROOT / "resultados" / "linhas" / "consumo_cardapio_calendario" / "04_competicao_modelos_ensembles" / "previsoes_modelos_ensembles_baselines.csv", low_memory=False)
historical = historical.loc[historical["modelo"].isin(manifest["componentes"])].copy()
historical["data"] = pd.to_datetime(historical["data"],format="mixed")
wide = historical.pivot(index="data",columns="modelo",values="previsao").reindex(columns=manifest["componentes"]).dropna()
matched = historical.drop_duplicates("data").set_index("data").reindex(wide.index)[["consumo_real"]].copy()
matched["previsao"] = wide.mean(axis=1)
matched["previsao_inteira"] = np.ceil(matched["previsao"])
matched["metodo"] = "Ensemble_fixo_cinco::media"
matched.reset_index().to_csv(RESULTS / "residuos_ensemble_fixo_historico.csv",index=False,encoding="utf-8-sig")
(RESULTS / "manifesto_diagnostico.json").write_text(json.dumps({"descricao":"Diagnóstico após examinar 2026; não é teste confirmatório independente.","arquivo_previsoes_recalculadas":str(source.relative_to(ROOT)),"sha256":source_hash,"n_datas_comuns":len(common),"treino_ate":"2025","margens":"Resíduos contínuos do mesmo conjunto de atributos e dos mesmos cinco componentes."},ensure_ascii=False,indent=2),encoding="utf-8")
print("Coorte comum",len(common),"sem modificar previsões congeladas.")


Folds semanais: 29 para pontuação histórica de 2026 e 1 para a semana de 5–9/10.
Executando sem reservas: Treino 2023–2025


Sem reservas | Treino 2023–2025: 8/30 origens


Sem reservas | Treino 2023–2025: 16/30 origens


Sem reservas | Treino 2023–2025: 24/30 origens


Sem reservas | Treino 2023–2025: 30/30 origens
Executando sem reservas: Treino somente 2025


...

Com reservas | Treino somente 2025: 30/30 origens


Ensemble congelado: 8/30 origens


Ensemble congelado: 16/30 origens


Ensemble congelado: 24/30 origens


Ensemble congelado: 30/30 origens
Previsões de ExtraTrees geradas: 750 linhas.
Coorte comum 123 sem modificar previsões congeladas.


In [3]:
# Margens por dia da semana calculadas apenas com erros fora da amostra de 2023–2025.
historical_pairs = [
    ("Sem reservas", ROOT / "resultados" / "linhas" / "consumo_cardapio_calendario" / "07_comparacao_reservas" / "previsoes_pareadas.csv", "Completo_Notebook04::ExtraTrees"),
    ("Com reservas", ROOT / "resultados" / "linhas" / "reservas_consumo_cardapio_calendario_clima" / "07_comparacao_reservas" / "previsoes_pareadas.csv", "Atual_mais_reservas::ExtraTrees"),
    ("Ensemble congelado", RESULTS / "residuos_ensemble_fixo_historico.csv", "Ensemble_fixo_cinco::media"),
]
weekday_names = {0: "segunda-feira", 1: "terça-feira", 2: "quarta-feira", 3: "quinta-feira", 4: "sexta-feira"}
margin_by_line = {}
for line, path, method in historical_pairs:
    hist = pd.read_csv(path, encoding="utf-8-sig")
    hist = hist.loc[hist["metodo"].eq(method)].copy()
    hist["data"] = pd.to_datetime(hist["data"], errors="raise")
    hist["erro_residual"] = pd.to_numeric(hist["consumo_real"], errors="raise") - pd.to_numeric(hist["previsao"], errors="raise")
    hist["dia_semana_num"] = hist["data"].dt.dayofweek
    q = hist.loc[hist["dia_semana_num"].lt(5)].groupby("dia_semana_num")["erro_residual"].quantile(2/3, interpolation="higher")
    margin_by_line[line] = q.clip(lower=0).to_dict()
    print(line, "margens não negativas Q67 por dia:", {weekday_names[k]: round(v, 1) for k, v in margin_by_line[line].items()})

scenario_metrics = []
scenario_forecasts = []
for case, group in predictions.groupby("caso", sort=False):
    scored = group.merge(actual[["data", "consumo"]], on="data", how="inner", validate="one_to_one")
    if scored.empty:
        continue
    line = "Com reservas" if case.startswith("Com reservas") else ("Ensemble congelado" if case.startswith("Ensemble congelado") else "Sem reservas")
    weekday = scored["data"].dt.dayofweek
    margin = weekday.map(margin_by_line[line]).fillna(0).to_numpy(dtype=float)
    scored["margem_q67_dia"] = margin
    scored["quantidade_pura"] = np.ceil(scored["previsao"].clip(lower=0)).astype(int)
    scored["quantidade_margem_10"] = np.ceil(scored["previsao"].clip(lower=0) + 10).astype(int)
    scored["quantidade_margem_q67_dia"] = np.ceil(scored["previsao"].clip(lower=0) + margin).astype(int)
    scored["residuo_previo"] = scored["consumo"].astype(float) - scored["previsao"].astype(float)
    scored["correcao_online_mediana_20"] = 0.0
    scored["correcao_online_q67_20"] = 0.0
    for idx, row in scored.iterrows():
        previous = scored.loc[scored["data"].lt(row["origem"])].sort_values("data").tail(20)["residuo_previo"]
        if len(previous) >= 5:
            scored.loc[idx, "correcao_online_mediana_20"] = float(previous.median())
            scored.loc[idx, "correcao_online_q67_20"] = float(previous.quantile(2/3, interpolation="higher"))
    scored["quantidade_calibrada_mediana_20"] = np.ceil((scored["previsao"] + scored["correcao_online_mediana_20"]).clip(lower=0)).astype(int)
    scored["quantidade_calibrada_q67_20"] = np.ceil((scored["previsao"] + scored["correcao_online_q67_20"]).clip(lower=0)).astype(int)
    for margin_name, quantity_col in [
        ("Pura", "quantidade_pura"),
        ("Margem fixa +10", "quantidade_margem_10"),
        ("Margem por dia Q67 histórica", "quantidade_margem_q67_dia"),
        ("Ajuste online pela mediana dos 20 erros", "quantidade_calibrada_mediana_20"),
        ("Plano pelo Q67 dos 20 erros recentes", "quantidade_calibrada_q67_20"),
    ]:
        error = scored[quantity_col].astype(float) - scored["consumo"].astype(float)
        scenario_metrics.append({
            "caso": case,
            "cenario": margin_name,
            "n_datas": len(scored),
            "MAE_refeicoes": float(error.abs().mean()),
            "WAPE_pct": float(error.abs().sum() / scored["consumo"].sum() * 100),
            "vies_refeicoes": float(error.mean()),
            "excesso_potencial_total": int(error.clip(lower=0).sum()),
            "falta_potencial_total": int((-error).clip(lower=0).sum()),
            "dias_com_falta": int((error < 0).sum()),
        })
    scored["linha"] = line
    scenario_forecasts.append(scored)

score_forecasts = pd.concat(scenario_forecasts, ignore_index=True)
metrics = pd.DataFrame(scenario_metrics)

# Compara, na mesma coorte, o modelo congelado anterior e a estimativa da nutricionista.
old_metrics = pd.read_csv(ROOT / "resultados" / "validacao_2026" / "metricas_2026_pareadas.csv", encoding="utf-8-sig")
for row in old_metrics.itertuples(index=False):
    metrics.loc[len(metrics)] = {
        "caso": "Validação congelada anterior",
        "cenario": row.metodo,
        "n_datas": row.n_datas,
        "MAE_refeicoes": row.MAE_refeicoes,
        "WAPE_pct": row.WAPE_pct,
        "vies_refeicoes": row.vies_refeicoes,
        "excesso_potencial_total": row.excesso_potencial_total,
        "falta_potencial_total": row.falta_potencial_total,
        "dias_com_falta": int(row.n_datas - row.dias_sem_deficit),
    }

reserve_silver = pd.read_csv(
    ROOT / "data" / "silver" / "reservas" / "reservas_diario_2023_2026.csv",
    encoding="utf-8-sig",
    parse_dates=["data"],
)
reserve_silver["data"] = pd.to_datetime(reserve_silver["data"], errors="raise").dt.normalize()
reserve_silver["reservas"] = pd.to_numeric(reserve_silver["reservas"], errors="coerce")
nutritionist = reserve_silver.loc[
    reserve_silver["data"].isin(actual["data"])
    & reserve_silver["status_reservas"].eq("observado"), ["data", "reservas"]
].dropna()
paired = actual[["data", "consumo"]].merge(nutritionist, on="data", how="inner", validate="one_to_one")
if len(paired) != len(actual):
    print(f"Atenção: reserva da nutricionista disponível em {len(paired)} de {len(actual)} datas; métricas comparativas serão pareadas.")
reserve_error = paired["reservas"].astype(float) - paired["consumo"].astype(float)
metrics.loc[len(metrics)] = {
    "caso": "Referência direta",
    "cenario": "Estimativa da nutricionista",
    "n_datas": len(paired),
    "MAE_refeicoes": float(reserve_error.abs().mean()),
    "WAPE_pct": float(reserve_error.abs().sum() / paired["consumo"].sum() * 100),
    "vies_refeicoes": float(reserve_error.mean()),
    "excesso_potencial_total": int(reserve_error.clip(lower=0).sum()),
    "falta_potencial_total": int((-reserve_error).clip(lower=0).sum()),
    "dias_com_falta": int((reserve_error < 0).sum()),
}

METRICS_PATH = RESULTS / "metricas_treino_pooled_vs_2025_only.csv"
PRED_PATH = RESULTS / "previsoes_pareadas_2026_e_semana_20261005.csv"
metrics.to_csv(METRICS_PATH, index=False, encoding="utf-8-sig")
score_forecasts.to_csv(PRED_PATH, index=False, encoding="utf-8-sig")
print()
print("Métricas comparativas (MAE, WAPE, viés, falta/excesso potencial):")
print(metrics.round(2).to_string(index=False))

Sem reservas

 margens não negativas Q67 por dia: {'segunda-feira': 10.7, 'terça-feira': 9.7, 'quarta-feira': 14.3, 'quinta-feira': 5.1, 'sexta-feira': 13.1}
Com reservas margens não negativas Q67 por dia: {'segunda-feira': 9.6, 'terça-feira': 8.8, 'quarta-feira': 13.9, 'quinta-feira': 4.8, 'sexta-feira': 11.7}
Ensemble congelado margens não negativas Q67 por dia: {'segunda-feira': 12.3, 'terça-feira': 11.2, 'quarta-feira': 15.4, 'quinta-feira': 7.6, 'sexta-feira': 13.4}



Métricas comparativas (MAE, WAPE, viés, falta/excesso potencial):
                                 caso                                 cenario  n_datas  MAE_refeicoes  WAPE_pct  vies_refeicoes  excesso_potencial_total  falta_potencial_total  dias_com_falta
      Sem reservas | Treino 2023–2025                                    Pura      123          36.00     21.53          -14.24                     1338                   3090              86
      Sem reservas | Treino 2023–2025                         Margem fixa +10      123          32.70     19.55           -4.24                     1750                   2272              75
      Sem reservas | Treino 2023–2025            Margem por dia Q67 histórica      123          32.28     19.30           -3.69                     1758                   2212              74
      Sem reservas | Treino 2023–2025 Ajuste online pela mediana dos 20 erros      123          37.57     22.46           -1.10                     2243             